In [3]:
from articles_dataset import ArticlesDataset

articles_dataset_path = \
    "../resources/bcc-news-entertainment.csv"
dataset = ArticlesDataset(articles_dataset_path)
dataset.load_articles(50)
dataset.preprocess()

sents_count = 0
for article in dataset.articles:
    sents_count += len(list(article.document.sents))
print("Total sentences count acrros loaded documents: ", sents_count)

Total sentences count acrros loaded documents:  743


In [4]:
import re
import hashlib


def readable_id(sentence: str, length: int = 4) -> str:
    h = hashlib.md5(sentence.encode()).hexdigest()[:length]
    slug = re.sub(r'\W+', '_', sentence.lower()).strip('_')[:30]

    return f"{slug}_{h}"

### NLTK based

In [5]:
import nltk
nltk.download('maxent_ne_chunker')

[nltk_data] Downloading package maxent_ne_chunker to /home/taras-
[nltk_data]     hamkalo/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!


True

In [15]:
from nltk.tree import Tree

def get_continuous_chunks(chunked):
    continuous_chunk = []
    current_chunk = []
    for i in chunked:
        if type(i) == Tree:
            current_chunk.append(" ".join([token for token, pos in i.leaves()]))

        if current_chunk:
            named_entity = " ".join(current_chunk)
            if named_entity not in continuous_chunk:
                if hasattr(i, 'label') and not i.label() == 'NE':
                    continuous_chunk.append((named_entity, i.label()))
                else:
                    continuous_chunk.append(named_entity)
                current_chunk = []
        else:
            continue

    return continuous_chunk

In [18]:
import os
import pandas as pd
# or load data by: nltk_ner.from_csv("../results/binary_ne_nltk.csv", index=False)
if os.path.exists("../results/ne_classification_nltk.csv") :
    nltk_nec = pd.read_csv("../results/ne_classification_nltk.csv")
else:
    rows = []
    for article in dataset.articles:
        for sentence in article.document.sents:
            # tokens = nltk.word_tokenize(sentence.text)
            tokens = [t.text for t in sentence]
            tagged = nltk.pos_tag(tokens)
            ne_chunked = nltk.ne_chunk(tagged, binary=False)
            continuous_chunk = get_continuous_chunks(ne_chunked)
            rows.append({
                "sentence_id": readable_id(sentence.text),
                "sentence": sentence.text,
                "tokens": tokens,
                "ne": continuous_chunk,
            })

nltk_nec = pd.DataFrame(rows)

In [20]:
nltk_nec.to_csv("../results/ne_classification_nltk.csv", index=False)
nltk_nec

,sentence_id,sentence,tokens,ne
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,"[ , French, musician, Jean, -, Michel, Jarre, ...","[(French, GPE), (Jean, ORGANIZATION), (Michel ..."
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,"[Denmark, is, holding, a, three, -, day, celeb...","[(Denmark, GPE), (Parken, ORGANIZATION)]"
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,"[Other, stars, are, expected, to, join, the, l...","[(Danish, GPE)]"
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...","["", Christian, Andersen, 's, fairy, tales, are...","[(Christian Andersen, PERSON), (Jarre, PERSON)]"
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...","["", For, all, of, us, ,, at, any, age, there, ...",[]
...,...,...,...,...
738,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,"[If, I, 'm, half, the, person, you, are, and, ...",[]
739,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,"[Both, Foxx, and, Swank, are, now, considered,...","[(Foxx, PERSON), (Swank, PERSON), (Oscars, PER..."
740,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...","[However, ,, Swank, has, to, overcome, a, stro...","[(Swank, PERSON), (Annette Bening, PERSON), (B..."
741,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...","[Meanwhile, ,, ballots, for, Oscars, -, the, H...","[(Oscars, GPE), (Hollywood, ORGANIZATION), (Ac..."


### Hugging face (Jean-Baptiste/roberta-large-ner-english)

In [21]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("Jean-Baptiste/roberta-large-ner-english")
model = AutoModelForTokenClassification.from_pretrained("Jean-Baptiste/roberta-large-ner-english")
nlp = pipeline('ner', model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: Jean-Baptiste/roberta-large-ner-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
import pandas as pd
rows = []
for article in dataset.articles:
    for sentence in article.document.sents:
        raw_ne = nlp(sentence.text)
        ne = [(ne["word"], ne["entity_group"]) for ne in raw_ne]
        rows.append({
            "sentence_id": readable_id(sentence.text),
            "sentence": sentence.text,
            "tokens": None,
            "ne": ne,
        })

model_ner = pd.DataFrame(rows)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [28]:
model_ner

,sentence_id,sentence,tokens,ne
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,None,"[( French, MISC), ( Jean-Michel Jarre, PER), (..."
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,None,"[( Denmark, LOC), ( Parken, LOC)]"
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,None,"[( Danish, MISC)]"
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...",None,"[(Christian Andersen, PER), ( Jarre, PER)]"
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",None,[]
...,...,...,...,...
738,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,None,[]
739,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,None,"[( Foxx, PER), ( Swank, PER), ( Oscars, MISC),..."
740,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...",None,"[( Swank, PER), ( Annette Bening, PER), ( Bein..."
741,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...",None,"[( Oscars, MISC), ( Hollywood, LOC), ( Academy..."


In [29]:
df = model_ner.merge(
    nltk_nec,
    on="sentence_id",
    suffixes=("_model", "_nltk")
)
df = df.drop(columns=["sentence_nltk", "tokens_model"])
df = df.rename(columns={"sentence_model": "sentence"})

In [30]:
df

,sentence_id,sentence,ne_model,tokens_nltk,ne_nltk
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,"[( French, MISC), ( Jean-Michel Jarre, PER), (...","[ , French, musician, Jean, -, Michel, Jarre, ...","[(French, GPE), (Jean, ORGANIZATION), (Michel ..."
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,"[( Denmark, LOC), ( Parken, LOC)]","[Denmark, is, holding, a, three, -, day, celeb...","[(Denmark, GPE), (Parken, ORGANIZATION)]"
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,"[( Danish, MISC)]","[Other, stars, are, expected, to, join, the, l...","[(Danish, GPE)]"
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...","[(Christian Andersen, PER), ( Jarre, PER)]","["", Christian, Andersen, 's, fairy, tales, are...","[(Christian Andersen, PERSON), (Jarre, PERSON)]"
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",[],"["", For, all, of, us, ,, at, any, age, there, ...",[]
...,...,...,...,...,...
744,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,[],"[If, I, 'm, half, the, person, you, are, and, ...",[]
745,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,"[( Foxx, PER), ( Swank, PER), ( Oscars, MISC),...","[Both, Foxx, and, Swank, are, now, considered,...","[(Foxx, PERSON), (Swank, PERSON), (Oscars, PER..."
746,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...","[( Swank, PER), ( Annette Bening, PER), ( Bein...","[However, ,, Swank, has, to, overcome, a, stro...","[(Swank, PERSON), (Annette Bening, PERSON), (B..."
747,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...","[( Oscars, MISC), ( Hollywood, LOC), ( Academy...","[Meanwhile, ,, ballots, for, Oscars, -, the, H...","[(Oscars, GPE), (Hollywood, ORGANIZATION), (Ac..."


In [31]:
df.to_csv("../results/ne_classification_nltk_vs_model.csv", index=False)